In [1]:
import pybedtools
from pybedtools import BedTool
import pandas as pd

A = pd.DataFrame({'chrom': ['chr1', 'chr1', 'chr1','chr1'],
                     'start': [1, 1, 5, 20],
                     'end': [10, 10, 7, 30],
					 'name': ['a1', 'a2', 'b', 'c']})

B = pd.DataFrame({'chrom': ['chr1', 'chr1', 'chr1', 'chr1'],
                     'start': [1, 1, 2, 4,],
                     'end': [2, 2, 3, 6,],
					 'name': ['d', 'e', 'f', 'g']})

A_bed = BedTool.from_dataframe(A)
B_bed = BedTool.from_dataframe(B)
print (A)
print (B)


  chrom  start  end name
0  chr1      1   10   a1
1  chr1      1   10   a2
2  chr1      5    7    b
3  chr1     20   30    c
  chrom  start  end name
0  chr1      1    2    d
1  chr1      1    2    e
2  chr1      2    3    f
3  chr1      4    6    g


In [2]:
B_bed.merge().to_dataframe()

,chrom,start,end
0,chr1,1,3
1,chr1,4,6


In [3]:
intersection = A_bed.intersect(B_bed, wb=True)
intersection.to_dataframe()

,chrom,start,end,name,score,strand,thickStart,thickEnd
0,chr1,1,2,a1,chr1,1,2,d
1,chr1,1,2,a1,chr1,1,2,e
2,chr1,2,3,a1,chr1,2,3,f
3,chr1,4,6,a1,chr1,4,6,g
4,chr1,1,2,a2,chr1,1,2,d
5,chr1,1,2,a2,chr1,1,2,e
6,chr1,2,3,a2,chr1,2,3,f
7,chr1,4,6,a2,chr1,4,6,g
8,chr1,5,6,b,chr1,4,6,g


In [4]:
df = intersection.to_dataframe()
agg_result = df.groupby('name').agg(
    thickStart_sum=('thickStart', 'sum'),
    span_sum=('end', lambda x: (x - df.loc[x.index, 'start']).sum())
)
agg_result

,thickStart_sum,span_sum
name,,
a1,13,5
a2,13,5
b,6,1


In [5]:
agg_result.reset_index()

,name,thickStart_sum,span_sum
0,a1,13,5
1,a2,13,5
2,b,6,1


In [ ]:
test1 = BedTool("data/test1.bed")
test2 = BedTool("data/test2.bed")
files = ["data/test1.bed", "data/test2.bed"]
result = test1.cat(*files, postmerge=False).sort().to_dataframe().groupby("name").apply(
	lambda x: BedTool.from_dataframe(x).merge().to_dataframe()
	).reset_index().drop(columns=['level_1'])
result

/tmp/ipykernel_3381637/3069698344.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test1.cat(*files, postmerge=False).sort().to_dataframe().groupby("name").apply(


,name,level_1,chrom,start,end
0,exon,0,chr1,5,15
1,exon,1,chr2,10,20
2,exon,2,chr2,30,40
3,intron,0,chr3,20,30


In [42]:
files[0]

'data/test1.bed'

In [43]:
!cat {files[0]}

chr1	5	10	exon
chr2	10	20	exon

In [45]:
BedTool(files[1]).to_dataframe()

,chrom,start,end,name
0,chr1,5,15,exon
1,chr2,30,40,exon


In [8]:
import pybedtools
from pybedtools import BedTool
!cat data/tests/test_features.bed | gzip > data/tests/test_features.bed.gz
!rm data/tests/test_features_symlinking.bed.gz
!ln -s test_features.bed.gz data/tests/test_features_symlinking.bed.gz
a = BedTool("data/tests/test_features.bed")
b = BedTool("data/tests/test_features.bed.gz")
c = BedTool("data/tests/test_features_symlinking.bed.gz")
print (c.to_dataframe())


  chrom  start  end    name
0  chr1      5   10    exon
1  chr1      7   15    exon
2  chr1     20   30  intron
3  chr1     20   25  intron
4  chr1     23   25    exon


In [6]:
import os
os.path.exists("data/tests/test_features_symlinking.bed.gz")

False

In [3]:
!ls -l data/tests/

total 20
-rw-r--r-- 1 fishman domain users  14 Jan  3 13:36 test_accrssible_inervals.bed
-rw-r--r-- 1 fishman domain users  81 Jan  3 13:37 test_features.bed
-rw-r--r-- 1 fishman domain users  64 Jan  6 08:50 test_features.bed.gz
lrwxrwxrwx 1 fishman domain users  28 Jan  6 08:50 test_features_symlinking.bed.gz -> data/tests/test_features.bed
-rw-r--r-- 1 fishman domain users 350 Jan  3 15:21 test_output.csv
-rw-r--r-- 1 fishman domain users  37 Jan  3 15:21 test_predictions.bed


# NTv3

In [4]:
# Note: to access NTv3, you need to login to Hugging Face first
# run:
# huggingface-cli login
# and enter your access token

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

repo = "InstaDeepAI/NTv3_650M_pre"
tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(repo, trust_remote_code=True)

batch = tok(["ATCGNATCG", "ACGT"], add_special_tokens=False, padding=True, pad_to_multiple_of=128, return_tensors="pt")
out = model(**batch)

print(out.logits.shape)  # (B, L, V = 11)


A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- tokenization_ntv3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- configuration_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- modeling_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


torch.Size([2, 128, 11])


In [12]:
batch["input_ids"][0]

tensor([ 6,  7,  8,  9, 10,  6,  7,  8,  9,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1])

In [10]:
batch["input_ids"][0].shape

torch.Size([128])

In [14]:
print (tok.sep_token_id)

None


In [15]:
print (tok.cls_token_id)

3


# EVO and Caduceus

In [ ]:
import h5py
h5file = "data/decoder-models/minja_human_chr21_only_hg38_evo2_mlm_embeddings.h5"
with h5py.File(h5file, 'r') as f:
    print(list(f.keys()))

['>chr21:10269868-10274327,10269868-10274327', '>chr21:10324327-10814560,10324327-10356327', '>chr21:10324327-10814560,10340327-10372327', '>chr21:10324327-10814560,10356327-10388327', '>chr21:10324327-10814560,10372327-10404327', '>chr21:10324327-10814560,10388327-10420327', '>chr21:10324327-10814560,10404327-10436327', '>chr21:10324327-10814560,10420327-10452327', '>chr21:10324327-10814560,10436327-10468327', '>chr21:10324327-10814560,10452327-10484327', '>chr21:10324327-10814560,10468327-10500327', '>chr21:10324327-10814560,10484327-10516327', '>chr21:10324327-10814560,10500327-10532327', '>chr21:10324327-10814560,10516327-10548327', '>chr21:10324327-10814560,10532327-10564327', '>chr21:10324327-10814560,10548327-10580327', '>chr21:10324327-10814560,10564327-10596327', '>chr21:10324327-10814560,10580327-10612327', '>chr21:10324327-10814560,10596327-10628327', '>chr21:10324327-10814560,10612327-10644327', '>chr21:10324327-10814560,10628327-10660327', '>chr21:10324327-10814560,1064432

In [ ]:
h5file = "data/decoder-models/minja_human_chr21_only_hg38_evo2_mlm_embeddings.h5"

with h5py.File(h5file, 'r') as f:
	keys = list(f.keys())
	for i in range(10):
		k = keys[i]
		lgts = f[k]["logits"][:]
		coords = k.split(",")[-1].split("-")
		print (k, lgts.shape, int(coords[0])-int(coords[1]))

>chr21:10269868-10274327,10269868-10274327 (1, 4459, 512) -4459
>chr21:10324327-10814560,10324327-10356327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10340327-10372327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10356327-10388327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10372327-10404327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10388327-10420327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10404327-10436327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10420327-10452327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10436327-10468327 (1, 32000, 512) -32000
>chr21:10324327-10814560,10452327-10484327 (1, 32000, 512) -32000


In [20]:
h5file = "data/decoder-models/minja_human_chr21_only_hg38_caduceus_ph_mlm_embeddings.h5"

with h5py.File(h5file, 'r') as f:
	keys = list(f.keys())
	for i in range(10):
		k = keys[i]
		lgts = f[k]["logits"][:]
		coords = k.split(",")[-1].split("-")
		print (k, lgts.shape, int(coords[0])-int(coords[1]))

>chr21:10269868-10274327,10269868-10274327 (1, 4459, 16) -4459
>chr21:10324327-10814560,10324327-10356327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10340327-10372327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10356327-10388327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10372327-10404327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10388327-10420327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10404327-10436327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10420327-10452327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10436327-10468327 (1, 32000, 16) -32000
>chr21:10324327-10814560,10452327-10484327 (1, 32000, 16) -32000


In [7]:
tokens = {"A":65, "C":67, "G":71, "T":84}
tokid = lgts[0].argmax(axis=1)
tokid[:10]

array([67, 65, 84, 84, 71, 65, 65, 65, 65, 65])

In [27]:
import torch
l = lgts[0]
print (l.shape)

from scipy.stats import entropy
e = entropy(torch.softmax(torch.from_numpy(l), dim=1).numpy(), axis=1)
print (e.shape, e[0])

from torch.distributions import Categorical
distr = Categorical(logits=torch.from_numpy(l).to(torch.device("cuda" if torch.cuda.is_available() else "cpu")))
e = distr.entropy()
print (e.shape, e[0])

(4459, 512)
(4459,) 1.3831687
torch.Size([4459]) tensor(1.3828, device='cuda:0', dtype=torch.float16)


In [18]:
torch.from_numpy(l).size()

torch.Size([4459, 512])

# Caduceus

In [1]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

# See the `Caduceus` collection page on the hub for list of available models.
model_name = "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)

/disk/10tb/home/fishman/miniconda3/envs/caduceus_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/disk/10tb/home/fishman/miniconda3/envs/caduceus_env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
tokenizer.special_tokens_map

{'bos_token': '[BOS]',
 'eos_token': '[SEP]',
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [3]:
inputs = tokenizer(
                "ATGCxATGC",
                padding="max_length",
                max_length=10,
                truncation=True,
                add_special_tokens=False
            )


In [4]:
type(inputs["input_ids"])

list

In [5]:
tokenizer.mask_token_id

3

In [6]:
tokenizer.sep_token_id

1

In [7]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

# See the `Caduceus` collection page on the hub for list of available models.
model_name = "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)

inputs = tokenizer(["AATATATATAGAAGATG"], padding=False, 
max_length=10, truncation=True, add_special_tokens=False,
return_tensors="pt")
model.cuda()
inputs = {k:v.to(model.device) for k,v in inputs.items()}
outs = model(**inputs)
print ("Vocab size:", outs["logits"].shape[-1])
try:
	print ("converting token 12:")
	tokenizer.convert_ids_to_tokens(12)
	print ("Success")
except Exception as e:
	# print error message and traceback
	print (e)
	import traceback
	traceback.print_exc()

/disk/10tb/home/fishman/miniconda3/envs/caduceus_env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Vocab size: 16
converting token 12:
12


Traceback (most recent call last):
  File "/tmp/ipykernel_270299/527367844.py", line 17, in <module>
    tokenizer.convert_ids_to_tokens(12)
  File "/disk/10tb/home/fishman/miniconda3/envs/caduceus_env/lib/python3.8/site-packages/transformers/tokenization_utils.py", line 973, in convert_ids_to_tokens
    return self._convert_id_to_token(ids)
  File "/disk/10tb/home/fishman/.cache/huggingface/modules/transformers_modules/kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16/b0477522ac5d044ad03578aa724ec8e4bdbd405b/tokenization_caduceus.py", line 97, in _convert_id_to_token
    return self._vocab_int_to_str[index]
KeyError: 12


In [8]:
import numpy as np
tokenizer.convert_ids_to_tokens(np.array([0,1,2]))

['[CLS]', '[SEP]', '[BOS]']

In [11]:
a = np.array([[1]*10,[2]*10])
a

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]])

In [14]:
highest_probs = np.max(a, axis=1)
highest_prob_token_ids = np.argmax(a, axis=1)
highest_prob_token_ids

array([0, 0])

In [23]:
highest_prob_tokens = []
for tok in highest_prob_token_ids.tolist():
	print (tok)
	try:
		highest_prob_tokens.append(tokenizer.convert_ids_to_tokens(tok))
	except KeyError as e:
		print (f"token id {tok} not found in tokenizer")
		highest_prob_tokens.append("P")


0
0


In [2]:
highest_prob_tokens

NameError: name 'highest_prob_tokens' is not defined

# tokenizer

In [6]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("AIRI-Institute/gena-lm-bert-base-t2t")
tok("ATGCTATATGATGACATGATC", return_offsets_mapping=True, add_special_tokens=True, padding="max_length", max_length=15)

/disk/10tb/home/fishman/miniconda3/envs/bert24/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'input_ids': [1, 71, 15424, 225, 36, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'offset_mapping': [(0, 0), (0, 4), (4, 13), (13, 18), (18, 21), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0)]}

In [3]:
input_ids = list(range(10))
meaningful_seq_len = 3
for i in range(0, len(input_ids) - meaningful_seq_len + 1, meaningful_seq_len):
	print (i)
	print (input_ids[i:i+meaningful_seq_len])


0
[0, 1, 2]
3
[3, 4, 5]
6
[6, 7, 8]


In [8]:
import numpy as np
np.array_split(np.arange(10), 10)

[array([0]),
 array([1]),
 array([2]),
 array([3]),
 array([4]),
 array([5]),
 array([6]),
 array([7]),
 array([8]),
 array([9])]

In [12]:
import torch
a = torch.arange(18).reshape(3,6)
a

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17]])

In [19]:
b = torch.max(a, dim=1).values
b

tensor([ 5, 11, 17])

In [20]:
b.masked_fill_(b<10, 10)
b

tensor([10, 11, 17])

In [22]:
c = b.clone()
c.masked_fill_(c<100, -1)
c


tensor([-1, -1, -1])

In [23]:
b

tensor([10, 11, 17])

In [ ]:
b.masked_fill_(b>0, -1)
b

SyntaxError: invalid syntax (436583413.py, line 1)

In [25]:
np.nonzero([True,False,True])

(array([0, 2]),)

In [45]:
test = torch.tensor([[2,3,4,5,2,7,8,9,10,2], [2,3,4,5,2,7,8,9,10,2]])
i  = 0
_num_splits = 2
special_tokens_mask = (test == 2).bool()
print (~special_tokens_mask[i])
print (np.nonzero(~special_tokens_mask[i].numpy())[0])
nonspecial_tokens_in_sequence = np.nonzero(~special_tokens_mask[i].numpy())[0] 
splits = np.array_split(nonspecial_tokens_in_sequence, _num_splits)
splits = [nonspecial_tokens_in_sequence[np.arange(len(split))*_num_splits + i] for i, split in enumerate(splits)]
splits

tensor([False,  True,  True,  True, False,  True,  True,  True,  True, False])
[1 2 3 5 6 7 8]


[array([1, 3, 6, 8]), array([2, 5, 7])]

In [49]:
test = torch.tensor([[1,2,3],[4,5,6]])
torch.repeat_interleave(test, torch.tensor([2,3]), dim=0)

tensor([[1, 2, 3],
        [1, 2, 3],
        [4, 5, 6],
        [4, 5, 6],
        [4, 5, 6]])

In [51]:
torch.zeros((2,3), dtype=torch.bool)

tensor([[False, False, False],
        [False, False, False]])

# data collator

In [92]:
from transformers import DataCollatorForLanguageModeling, default_data_collator
from typing import Optional, Tuple, Union, Any, Dict, List
import logging
import transformers
import torch
from typing import Mapping

class DataCollatorForMaskingAllPositions(transformers.DataCollatorForLanguageModeling):
	def tf_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
		raise NotImplementedError("TF is not supported for MLM with MLM probs")
						
	def torch_mask_tokens(self, inputs: Any, 
						  special_tokens_mask: Optional[Any] = None,
						  ) -> Tuple[Any, Any, Any]:
		"""
		Prepare masked tokens inputs/labels for masked language modeling.
		Based on the original implementation of transformers' `DataCollatorForLanguageModeling.torch_mask_tokens`

		given the batched-input and masking fraction, create super-batch where we mask all positions in the input, 
		mask_fraction of the tokens in each split are masked.
		return the super-batch and the labels array.
		"""
		# TODO: check if any corrections are needed to tenzors that are not 2D
		assert len(inputs.shape) == 2, f"inputs must be a 2D tensor, bsize * input_len, but got {inputs.shape}"
		labels = inputs.clone()

		if special_tokens_mask is None:
			special_tokens_mask = [
				self.tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
			]
			special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
		else:
			special_tokens_mask = special_tokens_mask.bool()
		
		Num_non_special_tokens = (~special_tokens_mask).sum(dim=1).float()
		assert (Num_non_special_tokens > 0).all(), f"Num_non_special_tokens must be greater than 0, but got {Num_non_special_tokens}"

		masking_fraction = self.mlm_probability
		assert masking_fraction < 1.0, f"Masking fraction must be less than or equal to 1.0, but got {masking_fraction}"
		assert masking_fraction > 0.0, f"Masking fraction must be greater than 0.0, but got {masking_fraction}"

		num_splits = int(1/masking_fraction)
		assert num_splits > 0, "Number of splits must be greater than 0"
		masked_token_ids = []
		n_repeats = []
		for i in range(inputs.shape[0]):
			nonspecial_tokens_in_sequence = np.nonzero(~special_tokens_mask[i].numpy())[0]
			_num_splits = num_splits
			if _num_splits > len(nonspecial_tokens_in_sequence):
				_num_splits = len(nonspecial_tokens_in_sequence)
			else:
				_num_splits = num_splits
			print (_num_splits)
			splits = np.array_split(nonspecial_tokens_in_sequence, _num_splits)
			assert len(splits) >= 1, f"Number of splits must be greater than or equal to 1, but got {len(splits)}"
			# evenly distribute tokens across splits
			splits = [nonspecial_tokens_in_sequence[np.arange(len(split))*_num_splits + i] for i, split in enumerate(splits)]
			n_repeats.append(len(splits))
			masked_token_ids.extend(splits)

		n_repeats = torch.tensor(n_repeats, dtype=torch.long)
		masked_indices = torch.zeros((n_repeats.sum().item(), inputs.shape[1]), dtype=bool)
		for sample_id in range(len(masked_token_ids)):
			sample_masked_token_ids = masked_token_ids[sample_id]
			masked_indices[sample_id, sample_masked_token_ids] = True
	
		labels = torch.repeat_interleave(labels, n_repeats, dim=0)
		labels[~masked_indices] = -100  # We only compute loss on masked tokens

		inputs = torch.repeat_interleave(inputs, n_repeats, dim=0)

		# we replace masked input tokens with tokenizer.mask_token ([MASK])
		inputs[masked_indices] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)
		print ("Mask token id:", self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token))
		print ("Masked indices:", masked_indices.sum())

		return inputs, labels, n_repeats
	
	def numpy_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
		raise NotImplementedError("Numpy is not supported for MLM with MLM probs")

	def numpy_mask_tokens(self, inputs: Any, special_tokens_mask: Optional[Any] = None) -> Tuple[Any, Any]:
		raise NotImplementedError("Numpy is not supported for MLM with MLM probs")

	def torch_call(self, examples: List[Union[List[int], Any, Dict[str, Any]]]) -> Dict[str, Any]:
		# Handle dict or lists with proper padding and conversion to tensor.
		if isinstance(examples[0], Mapping):
			# we don't need padding here, because we do pad ourselves in the dataset
			# batch = pad_without_fast_tokenizer_warning(
			#     self.tokenizer, examples, return_tensors="pt", pad_to_multiple_of=self.pad_to_multiple_of
			# )
			
			# handle mlm_efficiency_path
			if "mlm_efficiency_path" in examples[0]:
				mlm_efficiency_paths = [example.pop("mlm_efficiency_path") for example in examples]
			else:
				mlm_efficiency_paths = None

			keep_for_model = ["input_ids", "attention_mask", "offset_mapping_starts", "offset_mapping_ends", 
				"labels", "shard_id", "shard_sample_id"]
			batch = [{k: v for k, v in example.items() if k in keep_for_model} for example in examples]
			
			# now handle all tensors
			batch = default_data_collator(batch, self.return_tensors)			
		else:
			raise NotImplementedError("Only dicts are supported for MLM with MLM probs")
			# batch = {
			#     "input_ids": _torch_collate_batch(examples, self.tokenizer, pad_to_multiple_of=self.pad_to_multiple_of)
			# }

		# If special token mask has been preprocessed, pop it from the dict.
		special_tokens_mask = batch.pop("special_tokens_mask", None)
		batch["input_ids"], batch["labels"], n_repeats = self.torch_mask_tokens(
					batch["input_ids"], special_tokens_mask=special_tokens_mask
		)

		for k, v in batch.items():
			if not k in ["input_ids", "labels"]:
				batch[k] = torch.repeat_interleave(v, n_repeats, dim=0) # repeat the tensor along the first dimension

		# and keep the mlm_efficiency_path as single value in the batch
		if mlm_efficiency_paths is not None: 
			batch["mlm_efficiency_path"] = mlm_efficiency_paths

		return batch

seq = "ATAATGTAVTVATVGATATAGCAACGATGACATATATAGGATGCAGTGACTGACT"
args = {
	"max_length":20, 
	"truncation":True, 
	"add_special_tokens":True, 
	"padding":"max_length", 
	# "return_tensors":"pt"
}
s1 = tok(seq, **args)
s2 = tok("".join(np.random.permutation(list(seq))), **args)
batch = [s1,s2]
print (batch)
collator = DataCollatorForMaskingAllPositions(tokenizer=tok, mlm_probability=0.4)
collator(batch)["input_ids"]

[{'input_ids': [1, 54, 9088, 0, 15, 0, 27, 0, 9, 3931, 308, 410, 15708, 5493, 970, 15, 2, 3, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]}, {'input_ids': [1, 4097, 1943, 468, 187, 5620, 0, 9, 263, 75, 2154, 0, 636, 574, 0, 44, 2, 3, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]}]
2
2
Mask token id: 4
Masked indices: tensor(24)


tensor([[    1,     4,  9088,     0,     4,     0,    27,     0,     4,  3931,
             4,   410,     4,  5493,     4,    15,     2,     3,     3,     3],
        [    1,    54,     4,     0,    15,     0,     4,     0,     9,     4,
           308,     4, 15708,     4,   970,     4,     2,     3,     3,     3],
        [    1,     4,  1943,     4,   187,     4,     0,     9,     4,    75,
             4,     0,   636,     4,     0,    44,     2,     3,     3,     3],
        [    1,  4097,     4,   468,     4,  5620,     0,     4,   263,     4,
          2154,     0,     4,   574,     0,     4,     2,     3,     3,     3]])